In [ ]:
# lseg.data取得 -> weekly_intensity 出力まで（pipeline抜粋）

from pathlib import Path
import sys
import pandas as pd
import lseg.data as ld

PROJECT_ROOT = Path("/Users/kencharoff/workspace/projects/thematic-topic")
PROJECT_SRC = PROJECT_ROOT / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.append(str(PROJECT_SRC))

from thematic_topic.config import load_config, PipelineConfig
from thematic_topic.headlines import fetch_headlines
from thematic_topic.preprocess import clean_headlines
from thematic_topic.embeddings import encode_texts_with_cache
from thematic_topic.dedup import deduplicate_headlines
from thematic_topic.topics import fit_topic_model, transform_topics
from thematic_topic.intensity import build_topic_intensity


def _select_fit_sample(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")

    fit_start = cfg.topic_model.fit_start
    fit_end = cfg.topic_model.fit_end
    if fit_start:
        out = out[out["timestamp"] >= pd.to_datetime(fit_start, utc=True)]
    if fit_end:
        out = out[out["timestamp"] <= pd.to_datetime(fit_end, utc=True)]

    if out.empty:
        raise ValueError("BERTopic学習期間に該当するニュースがありません。")
    return out.reset_index(drop=True)


cfg = load_config(path=PROJECT_ROOT / "config/default.yaml", project_root=PROJECT_ROOT)

ld.open_session()
try:
    # 1) LSEG取得 + 正規化
    raw_df, normalized_df = fetch_headlines(cfg)

    # 2) 前処理
    clean_df = clean_headlines(
        normalized_df,
        mode=cfg.preprocess.low_information_mode,
        min_chars=cfg.preprocess.min_headline_chars,
    )

    # 3) embedding（本文主入力: model_input_text）
    cache_path = cfg.resolve_path(cfg.embedding.headline_cache_path)
    emb_meta, all_embeddings = encode_texts_with_cache(
        ids=clean_df["news_id"],
        texts=clean_df["model_input_text"],
        cache_path=cache_path,
        model_name=cfg.embedding.model_name,
        batch_size=cfg.embedding.batch_size,
        normalize_embeddings=cfg.embedding.normalize_embeddings,
    )

    # 4) 重複抑制
    dedup_df, dedup_log = deduplicate_headlines(
        clean_df,
        embeddings=all_embeddings,
        similarity_threshold=cfg.dedup.similarity_threshold,
        window_hours=cfg.dedup.window_hours,
    )

    # 5) dedup後embedding再マッピング
    clean_pos = clean_df.reset_index()[["index", "news_id"]]
    dedup_indexer = (
        dedup_df[["news_id"]]
        .merge(clean_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    dedup_emb = all_embeddings[dedup_indexer]

    # 6) BERTopic fit/transform
    fit_df = _select_fit_sample(dedup_df, cfg)
    dedup_pos = dedup_df.reset_index()[["index", "news_id"]]
    fit_idx = (
        fit_df[["news_id"]]
        .merge(dedup_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    fit_emb = dedup_emb[fit_idx]

    topic_model, fit_tables = fit_topic_model(
        fit_df, fit_emb, cfg.topic_model, text_col="model_input_text"
    )
    topic_assignments = transform_topics(
        dedup_df, topic_model, dedup_emb, text_col="model_input_text"
    )

    # 7) topic強度（daily/weekly）
    daily_intensity, weekly_intensity, outlier_stats = build_topic_intensity(
        topic_assignments,
        weekly_rule=cfg.topic_intensity.weekly_rule,       # "W-FRI"
        ewma_span=cfg.topic_intensity.ewma_span,           # 5
        lookback=cfg.topic_intensity.lookback,             # 20
        z_threshold=cfg.topic_intensity.z_threshold,       # 1.0
        aggregate_timezone=cfg.topic_intensity.aggregate_timezone,  # "Asia/Tokyo"
    )
finally:
    ld.close_session()

print("weekly_intensity shape:", weekly_intensity.shape)
display(weekly_intensity.head(20))
display(outlier_stats)


In [ ]:
# =========================
# 可視化ブロック: weekly / 内訳 / クラスタ
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

# ---------- 入力チェック ----------
required_weekly_cols = {"date", "topic_id", "topic_prob_sum", "topic_z", "topic_persistence"}
missing_weekly = sorted(required_weekly_cols - set(weekly_intensity.columns))
if missing_weekly:
    raise ValueError(f"weekly_intensity 必須カラム不足: {missing_weekly}")

required_assign_cols = {"topic_id"}
missing_assign = sorted(required_assign_cols - set(topic_assignments.columns))
if missing_assign:
    raise ValueError(f"topic_assignments 必須カラム不足: {missing_assign}")

# topic_summary の解決（任意）
topic_summary_df = None
if "topic_summary" in globals() and isinstance(topic_summary, pd.DataFrame):
    topic_summary_df = topic_summary
elif "topic_tables" in globals() and isinstance(topic_tables, dict) and "topic_summary" in topic_tables:
    topic_summary_df = topic_tables["topic_summary"]

# ---------- 1) weekly intensity 時系列 ----------
w = weekly_intensity.copy()
w["date"] = pd.to_datetime(w["date"], errors="coerce")
w = w.dropna(subset=["date"]).sort_values(["topic_id", "date"])

top_topics = (
    w.groupby("topic_id")["topic_prob_sum"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for tid in top_topics:
    g = w[w["topic_id"] == tid]
    axes[0].plot(g["date"], g["topic_prob_sum"], marker="o", linewidth=1.8, label=f"topic {tid}")
    axes[1].plot(g["date"], g["topic_z"], marker="o", linewidth=1.8, label=f"topic {tid}")

axes[0].set_title("Weekly topic_prob_sum (Top 5 topics)")
axes[1].set_title("Weekly topic_z (Top 5 topics)")
axes[0].legend(loc="upper left")
axes[1].legend(loc="upper left")
plt.tight_layout()
plt.show()

if "outlier_stats" in globals():
    print("outlier_stats:")
    display(outlier_stats)

# ---------- 2) topic内訳テーブル + 棒グラフ ----------
topic_count = (
    topic_assignments.groupby("topic_id", as_index=False)
    .size()
    .rename(columns={"size": "topic_count"})
)

prob_total = (
    w.groupby("topic_id", as_index=False)["topic_prob_sum"]
    .sum()
    .rename(columns={"topic_prob_sum": "prob_sum_total"})
)

latest_metrics = (
    w.sort_values("date")
    .groupby("topic_id", as_index=False)
    .tail(1)[["topic_id", "topic_ewma", "topic_z", "topic_persistence"]]
    .rename(
        columns={
            "topic_ewma": "latest_topic_ewma",
            "topic_z": "latest_topic_z",
            "topic_persistence": "latest_topic_persistence",
        }
    )
)

topic_breakdown = (
    topic_count.merge(prob_total, on="topic_id", how="outer")
    .merge(latest_metrics, on="topic_id", how="left")
    .sort_values("prob_sum_total", ascending=False)
    .reset_index(drop=True)
)

if topic_summary_df is not None and {"topic_id", "top_words"}.issubset(topic_summary_df.columns):
    tmp = topic_summary_df[["topic_id", "top_words"]].copy()
    tmp["top_words"] = tmp["top_words"].map(lambda x: ", ".join(x) if isinstance(x, list) else str(x))
    topic_breakdown = topic_breakdown.merge(tmp, on="topic_id", how="left")

print("topic_breakdown:")
display(topic_breakdown)

plt.figure(figsize=(10, 4.5))
plot_df = topic_breakdown.head(10).copy()
plt.bar(plot_df["topic_id"].astype(str), plot_df["prob_sum_total"])
plt.title("Topic Breakdown (prob_sum_total, Top 10)")
plt.xlabel("topic_id")
plt.ylabel("prob_sum_total")
plt.tight_layout()
plt.show()

# ---------- 3) クラスタ可視化 (UMAP -> fallback PCA) ----------
if "dedup_emb" not in globals():
    print("dedup_emb がないためクラスタ可視化をスキップします。")
else:
    emb = np.asarray(dedup_emb, dtype=float)
    if len(emb) != len(topic_assignments):
        print(
            f"len(dedup_emb) != len(topic_assignments) のためクラスタ可視化をスキップ: "
            f"{len(emb)} vs {len(topic_assignments)}"
        )
    else:
        topic_id_num = pd.to_numeric(topic_assignments["topic_id"], errors="coerce")
        valid_mask = topic_id_num.ne(-1).to_numpy()
        excluded = int((~valid_mask).sum())
        print(f"cluster plot: topic_id=-1 を除外しました（{excluded}件）")

        if valid_mask.sum() < 2:
            print("有効データが2件未満のためクラスタ可視化をスキップします。")
        else:
            emb_valid = emb[valid_mask]
            topics_valid = topic_assignments.loc[valid_mask, "topic_id"].to_numpy()

            reducer_name = "UMAP"
            try:
                from umap import UMAP
                reducer = UMAP(
                    n_neighbors=15,
                    n_components=2,
                    min_dist=0.0,
                    metric="cosine",
                    random_state=42,
                )
                coords = reducer.fit_transform(emb_valid)
            except Exception as e:
                from sklearn.decomposition import PCA
                reducer_name = f"PCA fallback ({e.__class__.__name__})"
                coords = PCA(n_components=2, random_state=42).fit_transform(emb_valid)

            cluster_df = pd.DataFrame(
                {"x": coords[:, 0], "y": coords[:, 1], "topic_id": topics_valid}
            )

            # 3-1) 散布図
            plt.figure(figsize=(8, 6))
            sns.scatterplot(
                data=cluster_df,
                x="x",
                y="y",
                hue="topic_id",
                palette="tab10",
                s=30,
                alpha=0.8,
                linewidth=0,
            )
            plt.title(f"Topic Cluster Scatter ({reducer_name})")
            plt.xlabel("dim-1")
            plt.ylabel("dim-2")
            plt.legend(title="topic_id", bbox_to_anchor=(1.02, 1), loc="upper left")
            plt.tight_layout()
            plt.show()

            # 3-2) トピック重心の類似度ヒートマップ
            uniq_topics = pd.Index(sorted(pd.unique(topics_valid)))
            centroids = []
            for tid in uniq_topics:
                centroids.append(emb_valid[topics_valid == tid].mean(axis=0))
            centroids = np.vstack(centroids)

            sim = cosine_similarity(centroids)
            sim_df = pd.DataFrame(
                sim,
                index=[f"topic {t}" for t in uniq_topics],
                columns=[f"topic {t}" for t in uniq_topics],
            )

            plt.figure(figsize=(7, 6))
            sns.heatmap(sim_df, cmap="YlGnBu", vmin=0, vmax=1, annot=True, fmt=".2f")
            plt.title("Topic Centroid Cosine Similarity")
            plt.tight_layout()
            plt.show()
